# Paired group-level bootstrap for feature-importance differences

Tests whether the five leading predictors are statistically separable from bedtime and average gaming time.

**Method:** one global resample of whole answer-pattern groups per bootstrap replicate; the same group counts are applied to every seed, fold and feature, so differences are paired and the interval is not artificially narrowed by averaging independent resamples. Importance = drop in average precision when a question is shuffled.

Run the cells **top to bottom**. You only need the dataset CSV (the Kaggle file).

## 1. Load the dataset

In [ ]:
DATA_FILE = None   # None = upload button in Colab; or set a path like "igd_data.csv"

import numpy as np, pandas as pd
if DATA_FILE is None:
    try:
        from google.colab import files
        up = files.upload()
        DATA_FILE = list(up)[0]
    except ImportError:
        raise SystemExit("Set DATA_FILE to the path of your CSV and run again.")
df = pd.read_csv(DATA_FILE)
print(df.shape)
print(list(df.columns))
df.head()

## 2. Build label, predictors and groups

- Label: `CGPA < 3.00` = at risk (1). Predictors: every other column. Groups: identical answer pattern = same group id.
- **Sanity checks against the paper:** 19 predictors, at-risk share about 0.297, about 605 groups. If any differ, your file or columns are not the ones used in the paper and the results will not match it.

In [ ]:
cgpa_col = [c for c in df.columns if "cgpa" in c.lower()]
assert len(cgpa_col) == 1, f"Found CGPA columns: {cgpa_col}. Set cgpa_col manually."
cgpa_col = cgpa_col[0]

y = (df[cgpa_col] < 3.00).astype(int).to_numpy()
X = df.drop(columns=[cgpa_col]).copy()
X = X.drop(columns=[c for c in X.columns if c.lower().startswith("unnamed")], errors="ignore")  # index column, not a predictor
groups = X.astype(str).agg("|".join, axis=1).factorize()[0]

print("predictors:", X.shape[1], "(paper: 19)")
print("at-risk share:", round(y.mean(), 3), "(paper: about 0.297)")
print("answer-pattern groups:", len(np.unique(groups)), "(paper: 605)")
assert X.shape[1] == 19, f"Expected 19 predictors, got {X.shape[1]}: {list(X.columns)}"
if X.shape[1] != 19 or len(np.unique(groups)) != 605:
    print("WARNING: does not match the paper's data. Check columns before trusting results.")

## 3. Choose the features to compare

Names are matched by keyword. **Read the printed mapping.** If a feature is missing or matched twice, the cell stops: put the exact column name in `OVERRIDE` and re-run.

In [ ]:
import re
KEYWORDS = {
    "HSC result":          r"higher\s*secondary|a\s*level",
    "Stress relief":       r"stress\s*relief",
    "Addictive game type": r"addict",
    "Platform (mobile/PC)":r"mobile\s*or\s*pc",
    "Newspaper reading":   r"newspaper",
    "Bedtime":             r"go\s*to\s*sleep",
    "Average gaming time": r"average\s*time",
}
OVERRIDE = {}   # e.g. {"Bedtime": "Sleep Time"}

mapping = {}
for label, pat in KEYWORDS.items():
    if label in OVERRIDE:
        mapping[label] = OVERRIDE[label]; continue
    hits = [c for c in X.columns if re.search(pat, c, flags=re.I)]
    mapping[label] = hits[0] if len(hits) == 1 else (hits if hits else None)

for k, v in mapping.items():
    print(f"{k:28s} -> {v}")
bad = [k for k, v in mapping.items() if not isinstance(v, str) or v not in X.columns]
assert not bad, f"Fix these in OVERRIDE (zero or several matches): {bad}\nColumns: {list(X.columns)}"

LEADERS = ["HSC result", "Stress relief", "Addictive game type", "Platform (mobile/PC)", "Newspaper reading"]  # check against Fig. 5 of the paper
REFS = ["Bedtime", "Average gaming time"]
INTEREST = [mapping[k] for k in LEADERS + REFS]

## 4. Fit the Random Forest per seed and fold, and store permuted predictions

Each original question is shuffled as one unit (before one-hot encoding), so Yes/No questions are not double-counted. `FAST = True` is a quick trial; use `False` for the paper settings.

In [ ]:
FAST = False
N_TREES, N_REPEATS, SEEDS = (100, 3, [0]) if FAST else (500, 10, [0, 1, 2, 3, 4])

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def fast_ap(y_true, score):
    # Average precision identical to sklearn's average_precision_score (tie-aware); checked against sklearn
    o = np.argsort(-score, kind="mergesort"); s = score[o]; yy = y_true[o]
    tp = np.cumsum(yy); n = np.arange(1, len(yy) + 1)
    last = np.r_[np.where(np.diff(s) != 0)[0], len(yy) - 1]
    tpt = tp[last]; prec = tpt / n[last]; rec = tpt / tp[-1]
    return float(np.sum(np.diff(np.r_[0, rec]) * prec))

cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
store, imp_rows = [], []
for seed in SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(cv.split(X, y, groups)):
        pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
                                remainder="passthrough")
        model = Pipeline([("pre", pre),
                          ("rf", RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt",
                                                        random_state=seed, n_jobs=-1))])
        model.fit(X.iloc[tr], y[tr])
        Xte = X.iloc[te].reset_index(drop=True); yte = y[te]; gte = groups[te]
        p0 = model.predict_proba(Xte)[:, 1]
        ap0 = fast_ap(yte, p0)
        prng = np.random.default_rng(1000 * seed + fold)
        perm = {}
        for feat in X.columns:
            ps = []
            for _ in range(N_REPEATS):
                Xp = Xte.copy()
                Xp[feat] = prng.permutation(Xp[feat].to_numpy())
                ps.append(model.predict_proba(Xp)[:, 1])
            ps = np.array(ps)
            imp_rows.append((seed, fold, feat, ap0 - np.mean([fast_ap(yte, p) for p in ps])))
            if feat in INTEREST:
                perm[feat] = ps
        store.append({"y": yte, "g": gte, "p0": p0, "perm": perm})
    print("seed", seed, "done")

imp = pd.DataFrame(imp_rows, columns=["seed", "fold", "feature", "importance"])
imp.to_csv("importances_per_fold.csv", index=False)
summary = imp.groupby("feature")["importance"].agg(["mean", "std"]).sort_values("mean", ascending=False)
summary.round(4)

## 5. Paired group-level bootstrap

Within each held-out fold, answer-pattern groups are resampled with replacement (same resample for all features).

In [ ]:
N_BOOT, RNG_SEED = (200 if FAST else 2000), 0

all_groups = np.unique(np.concatenate([u["g"] for u in store]))
gpos = {g: i for i, g in enumerate(all_groups)}
for u in store:
    u["gi"] = np.array([gpos[g] for g in u["g"]])
rng = np.random.default_rng(RNG_SEED)
boot = {f: np.full(N_BOOT, np.nan) for f in INTEREST}
for b in range(N_BOOT):
    # one global group resample per replicate, shared by every seed, fold and feature
    counts = np.bincount(rng.integers(0, len(all_groups), len(all_groups)), minlength=len(all_groups))
    acc = {f: [] for f in INTEREST}
    for u in store:
        idx = np.repeat(np.arange(len(u["y"])), counts[u["gi"]])
        yb = u["y"][idx]
        if yb.sum() == 0:
            continue
        base = fast_ap(yb, u["p0"][idx])
        for f in INTEREST:
            acc[f].append(base - np.mean([fast_ap(yb, p[idx]) for p in u["perm"][f]]))
    for f in INTEREST:
        boot[f][b] = np.mean(acc[f])

point = imp.groupby("feature")["importance"].mean()
out = []
for a in LEADERS:
    for r_ in REFS:
        fa, fr = mapping[a], mapping[r_]
        d = boot[fa] - boot[fr]
        lo, hi = np.nanpercentile(d, [2.5, 97.5])
        lo99, hi99 = np.nanpercentile(d, [0.5, 99.5])
        if lo99 > 0 or hi99 < 0:
            verdict = "separable"            # 99% interval excludes 0
        elif lo > 0 or hi < 0:
            verdict = "suggestive"           # only the 95% interval excludes 0
        else:
            verdict = "overlap"
        out.append({"feature": a, "vs": r_, "mean_diff": point[fa] - point[fr],
                    "ci95_lo": lo, "ci95_hi": hi, "ci99_lo": lo99, "ci99_hi": hi99,
                    "verdict": verdict})
result = pd.DataFrame(out)
print(result["verdict"].value_counts().to_dict())
result.round(4)

## 6. How to use the result

- `separable`: the **99%** interval of the paired difference excludes 0. Only for these pairs may you say the predictor has higher importance than that reference feature.
- `suggestive`: only the 95% interval excludes 0. Do not claim separation; at most say the difference is suggestive.
- `overlap`: keep the current paper wording (highest point estimates, ranking not statistically separable).
- Why 99%: ten pairs are tested, so a single 95% interval would flag too many pairs by chance. The 99% interval is a conservative rule (it is not an exact Bonferroni correction, which for ten pairs would be 99.5%). Say "conservative 99% interval" in the paper.
- Report like Table IV, e.g. "HSC result minus bedtime: +0.026 [0.010, 0.042]".

## 7. Save tables and figures (CSV + PNG)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
OUT = "paired_outputs"
import os; os.makedirs(OUT, exist_ok=True)

res = result.copy()
res["comparison"] = res["feature"] + " \u2212 " + res["vs"]
res.round(5).to_csv(f"{OUT}/paired_importance_differences.csv", index=False)
imp.to_csv(f"{OUT}/importances_per_fold.csv", index=False)
summary.round(5).to_csv(f"{OUT}/importance_summary_mean_sd.csv")
pd.DataFrame({"setting": ["FAST","N_TREES","N_REPEATS","SEEDS","N_BOOT","RNG_SEED","n_groups","n_rows"],
              "value": [FAST, N_TREES, N_REPEATS, str(SEEDS), N_BOOT, RNG_SEED, len(np.unique(groups)), len(y)]}
             ).to_csv(f"{OUT}/run_settings.csv", index=False)
bd = pd.DataFrame({mapping_name: boot[col] for mapping_name, col in zip(LEADERS+REFS, INTEREST)})
bd.to_csv(f"{OUT}/bootstrap_importance_draws.csv", index=False)

colors = {"separable": "#1f6f3d", "suggestive": "#c77d00", "overlap": "#6b6b6b"}
fig, ax = plt.subplots(figsize=(9, 0.45*len(res)+1.6))
yy = np.arange(len(res))[::-1]
for yv, (_, r) in zip(yy, res.iterrows()):
    c = colors[r["verdict"]]
    ax.plot([r["ci99_lo"], r["ci99_hi"]], [yv, yv], color=c, lw=1.5, alpha=.45)
    ax.plot([r["ci95_lo"], r["ci95_hi"]], [yv, yv], color=c, lw=4)
    ax.plot(r["mean_diff"], yv, "o", color="white", mec=c, mew=2, ms=7)
ax.axvline(0, color="black", ls="--", lw=1)
ax.set_yticks(yy); ax.set_yticklabels(res["comparison"], fontsize=9)
ax.set_xlabel("Difference in importance (decrease in PR-AUC); thick = 95%, thin = 99% paired group bootstrap")
ax.set_title("Leading predictors minus bedtime / average gaming time")
from matplotlib.lines import Line2D
ax.legend([Line2D([0],[0],color=v,lw=4) for v in colors.values()], list(colors), loc="lower right", fontsize=8, title="verdict")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_paired_importance_differences.png", dpi=200, bbox_inches="tight"); plt.show()

# importance bar chart with SD (descriptive), for reference
sm = summary.sort_values("mean")
fig, ax = plt.subplots(figsize=(8, 0.3*len(sm)+1.5))
ax.barh([f[:55] for f in sm.index], sm["mean"], xerr=sm["std"], capsize=2, color="#4a6fa5")
ax.axvline(0, color="black", ls="--", lw=1); ax.set_xlabel("Decrease in PR-AUC (mean \u00b1 SD across seed\u00d7fold results)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_importance_mean_sd.png", dpi=200, bbox_inches="tight"); plt.show()
print(sorted(os.listdir(OUT)))

## 8. Download everything as one ZIP

In [ ]:
import shutil, pathlib
files = sorted(p for p in pathlib.Path(OUT).rglob("*") if p.is_file())
print(f"Files in ZIP ({len(files)}):")
for p in files: print(" -", p.name)
zip_path = shutil.make_archive("paired_importance_outputs", "zip", root_dir=OUT)
print("Created:", zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except ImportError:
    print("Not in Colab; ZIP is in the working folder.")